In [ ]:
import sys, os
# ensure parent directory is on the path so `src` package can be imported
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
# configura per importare da src
import sys
sys.path.append('./src')

In [ ]:
from src.model import ConceptBoxModel
from src.train import train_step
from src.dataset import load_and_split_awa2_features
from torch import optim
import torch
import numpy as np
import pandas as pd

In [ ]:
# leggi features da file txt
with open('../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/AwA2-features.txt', 'r') as f:
    h_features = np.array([[float(num) for num in line.split()] for line in f])
h_features = torch.tensor(h_features, dtype=torch.float32)

In [ ]:
# leggi matrice V_gt da file csv
V_gt = pd.read_csv('../AwA2_Dataset_Labels/Animals_with_Attributes2/V_gt.csv', sep='\t', index_col=0).values
V_gt = torch.tensor(V_gt, dtype=torch.float32)

In [ ]:
features_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/AwA2-features.txt'
labels_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/AwA2-labels.txt'
classes_path = '../Awa2_Dataset_Labels/Animals_with_Attributes2/classes.txt'
train_split_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/trainclasses1.txt'
val_split_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/valclasses1.txt'
test_split_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/testclasses.txt'

(X_train, y_train), (X_val, y_val), (X_test, y_test) = load_and_split_awa2_features(
    features_path, labels_path, classes_path, 
    train_split_path, val_split_path, test_split_path
)

In [ ]:
class_concept_matrix = np.loadtxt('../Awa2_Dataset_Labels/Animals_with_Attributes2/extended_matrix.txt', dtype=int)

In [ ]:
LATENT_DIM = h_features.shape[1]
print(f"Dimensione features: {LATENT_DIM}")
NUM_CONCEPTS = V_gt.shape[0]  # numero di classi è dato dal numero di righe di V_gt
print(f"Numero di concetti: {NUM_CONCEPTS}")
NUM_CLASSES = len(set(y_train))  # numero di classi è dato dal numero di classi uniche nei label di training
print(f"Numero di classi: {NUM_CLASSES}")

# Iperparametri del modello
BOX_DIM = 16
BATCH_SIZE = 32
    
model = ConceptBoxModel(LATENT_DIM, NUM_CONCEPTS, BOX_DIM, NUM_CLASSES)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
# Dati fittizi (simulano le features estratte da Psi(x) e le ground truth)
concepts = torch.randint(0, 2, (BATCH_SIZE, NUM_CONCEPTS)).float()
    
# Esecuzione di uno step di training
total, l_t, l_c, l_b = train_step(model, optimizer, h_features, y_train, class_concept_matrix, V_gt)
    
print(f"Loss Totale: {total:.4f} (Task: {l_t:.4f}, Concept: {l_c:.4f}, Box: {l_b:.4f})")